# Netflix Dataset

Available on Kaggle at https://www.kaggle.com/datasets/shivamb/netflix-shows

To run the code it's necessary to import the following library.

*For one of the followind graph it's necessary to install OpenCage library, using the command ***pip install opencage****

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
#import plotly.io as pio
#pio.renderers.default='notebook'

### Data Read

Reading form the dataset from the *csv* file and and displaying some example rows.

In [3]:
netflix_df = pd.read_csv("netflix_titles.csv", low_memory=False)
netflix_df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


### Check missing data

Check in each column how many *null* values there are.

In [4]:
netflix_df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

### Deal with missing data

After read the dataset from *csv* file, in order to avoid wrong data, it's necessary to apply some manipulation to dataset.

In this case:
* Replace blank ***countries*** with the mode (most common) country.
* Replace ***nan*** values with *"No Data"* for ***cast*** and ***director*** columns.
* Convert the column ***release_year*** to numeric values.
* Extraction of numbers from the ***duration*** column, using a regular expression, to isolate the numerical part.
* Conversion of ***date_added*** column to datetime format.
* Extraction of the year from the ***date_added*** column and saving in the new ***year_added*** column.
* Extraction of the name of the month (e.g. "January", "February") from the ***date_added*** column and saves it in the ***month_name_added*** column.
* Rows containing invalid ratings are removed.

In [5]:
netflix_df['country'] = netflix_df['country'].fillna(netflix_df['country'].mode()[0])

netflix_df['cast'] = netflix_df['cast'].replace(np.nan, 'No Data')
netflix_df['director'] = netflix_df['director'].replace(np.nan, 'No Data')

netflix_df['release_year'] = pd.to_numeric(netflix_df['release_year'])
netflix_df['duration_minutes'] = netflix_df['duration'].str.extract(r'(\d+)').astype(float)

netflix_df['date_added'] = pd.to_datetime(netflix_df['date_added'], errors='coerce')
netflix_df['year_added'] = netflix_df['date_added'].dt.year
netflix_df['month_name_added'] = netflix_df['date_added'].dt.month_name()

valid_ratings = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'TV-PG', 'TV-14', 'TV-MA', 'NR', 'UR']
netflix_df = netflix_df[netflix_df['rating'].isin(valid_ratings)]

netflix_df.dropna(inplace=True)
netflix_df = netflix_df.drop_duplicates()

### Re-Check missing data

Now the dataset has no null value.

In [6]:
netflix_df.isnull().sum()

show_id             0
type                0
title               0
director            0
cast                0
country             0
date_added          0
release_year        0
rating              0
duration            0
listed_in           0
description         0
duration_minutes    0
year_added          0
month_name_added    0
dtype: int64

## 1° Graph

### Production trends of movies and TV shows over the years in **relation to historical events**.

This graph visualizes the trends in movie and TV show productions over the years, as recorded in the Netflix dataset. The *x-axis* represents the *release year*, while the *y-axis* shows the *number of productions* for each year. The data is segmented by the type of production (Movie or TV Show).
<br><br>
The graph also **highlights significant historical events** with vertical dashed lines, marking their impact on production trends. These events include:

In [7]:
# Prepare data for the production trend chart by year. Will be added a "count" column with the number of productions
production_counts = netflix_df.groupby(['release_year', 'type']).size().reset_index(name='count')

# Create a line chart
fig = px.line(
    production_counts,
    x='release_year',
    y='count',
    title=" ",
    labels={'release_year': 'Release Year', 'count': 'Number of Productions'},
    color='type',
    color_discrete_map={'Movie': '#b20710', 'TV Show': '#221f1f'}
)

# Define key historical events with their corresponding years
events = {
    "1997<br>Netflix Foundation": 1997,
    "2008<br>Financial Crisis": 2008,
    "2013<br>Rise of Streaming": 2013,
    "2020<br>COVID-19": 2020
}

# Add vertical lines for historical events
for event, year in events.items():
    fig.add_vline(
        x=year, 
        line=dict(color="blue", dash="dash"),
        annotation_text=event, 
        annotation_position="top", 
        annotation=dict(textangle=-45, font=dict(size=14))
    )

fig.update_layout(
    title_font=dict(size=25),
    xaxis=dict(
        title='Release Year'
    ),
    yaxis=dict(
        title='Number of Productions'
    ),
    legend=dict(
        title='Type'
    ),
    font=dict(size=16)
)

fig.show()


## 2° Graph

### Distribution of TV Shows and Movies added on Platform per Year

This bar chart displays the number of *TV shows* and *movies* added to Netflix each year, allowing for a **comparison of the distribution** between the two types of content over time.
<br><br>
Allowing to observe trends in content addition to Netflix over time, highlighting whether the platform has been more focused on *movies* or *TV shows* in certain years.

In [8]:
# Group the data by 'year_added' and 'type', and count the number of occurrences for each combination
added_per_year = netflix_df.groupby(['year_added', 'type']).size().reset_index(name='count')

# Create a bar chart to visualize the distribution of TV shows and movies added each year
fig = px.bar(
    added_per_year, x='year_added', y='count', color='type',
    title="Distribution of TV Shows and Movies added on Platform per Year",
    color_discrete_map={'Movie': '#b20710', 'TV Show': '#221f1f'}
)

fig.update_layout(
    height=600,
    title_font=dict(size=30),
    xaxis=dict(
        title='Added Year'
    ),
    yaxis=dict(
        title='Film and TV Shows added'
    ),
    legend=dict(
        title='Type'
    ),
    font=dict(size=16)
)

fig.show()

## 3° Graph

### Distribution of Film Duration per Year

This plot visualizes the distribution of film durations over the years for movies in the Netflix dataset. It uses a **box plot** to show the spread and central tendency of the data, and a **median line** to highlight the middle value for each year.

Providing an overview of the trends and distribution of movie lengths over time, offering insights into how the industry’s approach to film duration may have shifted.

In [9]:
# Group the data to calculate the average film duration per year for movies
average_duration_per_year = netflix_df[netflix_df['type'] == 'Movie'].groupby('release_year')['duration_minutes'].mean().reset_index()

# Create a box plot to show the distribution of film durations for each year
fig = px.box(
    netflix_df[netflix_df['type'] == 'Movie'], 
    x='release_year', 
    y='duration_minutes', 
    title='Distribution of Film Length per Year',
    color_discrete_sequence=['#b20710']
)

# Calculate the median film duration per year for movies
median_per_year = netflix_df[netflix_df['type'] == 'Movie'].groupby('release_year')['duration_minutes'].median().reset_index()

# Add a scatter plot for the median line, to indicate the median duration per year
fig.add_scatter(
    x=median_per_year['release_year'], 
    y=median_per_year['duration_minutes'], 
    mode='lines+markers', 
    name='Median', 
    line=dict(color='#221f1f')
)

fig.update_layout(
    title_font=dict(size=30),
    xaxis=dict(
        title='Year'
    ),
    yaxis=dict(
        title='Duration (minutes)'
    ),
    font=dict(size=16)
)

fig.show()

## 4° Graph

### Distribution of Film Duration

This histogram shows how **movie durations** are distributed in the dataset, indicating how many movies fall into different duration ranges (e.g., 0–19 minutes, 20-39 minutes, etc.).
<br><br>
Providing valuable information about the duration patterns in movies, offering insights into the typical lengths of films available on Netflix.

In [10]:
# Create a histogram to show the distribution of film durations for movies
fig = px.histogram(
    netflix_df[netflix_df['type'] == 'Movie'], 
    x='duration_minutes', 
    nbins=20,
    title="Distribution of Film Duration", 
    labels={'duration_minutes': 'Duration (minutes)'})

fig.update_traces(marker_color='#b20710')

fig.update_layout(
    bargap=0.1,
    title_font=dict(size=30),
    xaxis=dict(
        title='Duration (minutes)'
    ),
    yaxis=dict(
        title='Count'
    ),
    font=dict(size=16)
)

fig.show()

## 5° Graph

### Monthly Distribution of Movies and TV Shows by Hemisphere

This plot visualizes the distribution of Netflix Movies and TV Shows by hemisphere (North, South, or Both) across each month of the year. It consists of two key components:
1. Line Chart: displays the number of Movies and TV Shows added to Netflix each month, with data split by hemisphere.
2. Table of Countries by Hemisphere: displays the countries that belong to the North and South hemispheres.
<br>

The plot aims to give insights into how Netflix's content distribution (Movies and TV Shows) varies across different hemispheres and throughout the year. By combining the time-series line chart with the country listing, the plot offers both temporal and geographic perspectives on the content landscape of Netflix.

In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Dictionary defining countries in the Northern, Southern, and Both hemispheres
hemisphere_countries = {
    'North': ['United States', 'Canada', 'United Kingdom', 'France', 'Germany', 'Italy', 'Spain', 'India', 'China', 'Japan', 'Mexico', 'Russia', 'Turkey'],
    'South': ['Australia', 'Argentina', 'Brazil', 'South Africa', 'Chile', 'New Zealand'],
    'Both': ['Ecuador', 'Colombia', 'Kenya', 'Indonesia']
}

# Function to assign a hemisphere (North, South, Both, or Unknown) based on the country
def assign_hemisphere(country):
    if country in hemisphere_countries['North']:
        return 'North'
    elif country in hemisphere_countries['South']:
        return 'South'
    elif country in hemisphere_countries['Both']:
        return 'Both'
    return 'Unknown'

# Process the 'country' column to generate a list of countries for each production
netflix_df['country_list'] = netflix_df['country'].fillna('').apply(lambda x: [p.strip() for p in x.split(',') if p])

# Assign hemisphere based on the countries listed in the 'country_list' for each production
netflix_df['hemisphere'] = netflix_df['country_list'].apply(
    lambda x: 'North' if any(assign_hemisphere(p) == 'North' for p in x) else (
        'South' if any(assign_hemisphere(p) == 'South' for p in x) else (
        'Both' if any(assign_hemisphere(p) == 'Both' for p in x) else 'Unknown'))
)

# Filter the data to only include rows where the hemisphere is North or South and group by month, type, and hemisphere
distribution = netflix_df[netflix_df['hemisphere'].isin(['North', 'South'])].groupby(['month_name_added', 'type', 'hemisphere']).size().reset_index(name='count')

# Define the order of months for correct sorting in the plot
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
distribution['month_name_added'] = pd.Categorical(distribution['month_name_added'], categories=month_order, ordered=True)
distribution = distribution.sort_values('month_name_added')

fig = px.line(distribution, x='month_name_added', y='count', color='hemisphere', line_dash='type', color_discrete_sequence=['#b20710','#221f1f'],
              title='Monthly Distribution of Movies and TV Shows by Hemisphere',
              labels={'month_name_added': 'Month', 'count': 'Count', 'hemisphere': 'Hemisphere', 'type': 'Type'},
              category_orders={"month_name_added": month_order})

fig.update_traces(mode='lines+markers', visible="legendonly")

fig.update_layout(
    title_font=dict(size=30),
    xaxis_title="Month",
    yaxis_title="Count of Additions",
)

# Create a table to show the countries in each hemisphere (North and South)
table_data = go.Table(
    header=dict(values=["North Hemisphere", "South Hemisphere"],
                fill_color='lightgrey', align='center', font=dict(size=12)),
    cells=dict(values=[hemisphere_countries['North'], hemisphere_countries['South']],
               fill_color='lavender', align='left', 
               height=30, line=dict(color='darkslategray'), font=dict(size=11))
)

fig_final = make_subplots(
    rows=1, cols=2, column_widths=[0.7, 0.3],
    subplot_titles=("Monthly Distribution", "Countries by Hemisphere"),
    specs=[[{"type": "xy"}, {"type": "table"}]],
    horizontal_spacing=0.15
)

fig_final.add_traces(fig.data, rows=1, cols=1)
fig_final.add_trace(table_data, row=1, col=2)

fig_final.update_layout(
    title_font=dict(size=30),
    title_text="Monthly Distribution of Movies and TV Shows with Countries by Hemisphere",
    showlegend=True,
    legend=dict(title="Hemisphere", x=0.5, y=-0.2, orientation="h"),
    margin=dict(l=50, r=50, t=100, b=100),
    font=dict(size=15)
)

fig_final.show()

## 6° Graph

### Number of Films Produced in Each Country

This interactive map visualizes the **distribution of films produced across various countries**, based on data from the Netflix dataset. Each country is represented by a point on the map, *size* and *color* of the point corresponds to the number of films produced in that country.
<br><br>
This plot provides a visual summary of global film production, helping to identify key regions in the entertainment industry.

In [12]:
from opencage.geocoder import OpenCageGeocode
import os

# Initialize OpenCage API key and geocoder instance
api_key = '2192adbcd9c54f90a055ca132a3a242a'
geocoder = OpenCageGeocode(api_key)

# Define the cache file to store previously fetched coordinates
cache_file = 'country_coordinates_cache.csv'

# Load coordinates cache if it exists, or create a new DataFrame
if os.path.exists(cache_file):
    coordinates_cache = pd.read_csv(cache_file)
else:
    coordinates_cache = pd.DataFrame(columns=['country', 'latitude', 'longitude'])

# Function to fetch latitude and longitude for a given country, checking cache first
def get_lat_lon(country):
    # Look for cached coordinates for the country
    cached_entry = coordinates_cache[coordinates_cache['country'] == country]
    if not cached_entry.empty:
        return cached_entry.iloc[0]['latitude'], cached_entry.iloc[0]['longitude']
    
    # If not found in cache, fetch coordinates from the OpenCage API
    result = geocoder.geocode(country)
    if result:
        lat, lon = result[0]['geometry']['lat'], result[0]['geometry']['lng']
        
        # Save the new coordinates in the cache and update the cache file
        coordinates_cache.loc[len(coordinates_cache)] = [country, lat, lon]
        coordinates_cache.to_csv(cache_file, index=False)
        return lat, lon
    else:
        return None, None

# Count the occurrences of each country in the 'country' column
country_counts = netflix_df['country'].str.split(', ').explode().value_counts().reset_index()
country_counts.columns = ['country', 'count']
country_counts['country'] = country_counts['country'].str.strip()

# Apply the function to get latitude and longitude for each country
country_counts[['latitude', 'longitude']] = country_counts['country'].apply(
    lambda x: pd.Series(get_lat_lon(x))
)

# Remove countries where latitude or longitude could not be found
country_counts.dropna(subset=['latitude', 'longitude'], inplace=True)

# Create a Mapbox scatter plot showing the number of films produced in each country
fig = px.scatter_mapbox(
    country_counts,
    lat='latitude',
    lon='longitude',
    size='count',
    hover_name="country",
    size_max=50,
    color='count',
    color_continuous_scale="Plasma",
    title="Number of Films Produced in Each Country",
    mapbox_style="open-street-map",
    zoom=1.3,
    range_color=[0, 1500],
    center=dict(lat=10, lon=10),
)

fig.update_layout(
    height=1000,
    title_font=dict(size=30),
    coloraxis_colorbar=dict(title="Number of Films"),
    font=dict(size=16)
)

fig.show()


## 7° Graph

### Distribution of Netflix Ratings by Content Type

This bar chart illustrates the distribution of Netflix content across various rating categories, comparing Movies and TV Shows. The table on the right provides a reference for the rating categories and their associated subgroups, offering insight into the classification system for different age groups and content types.

In [13]:
# Create a 'count' column for counting occurrences
netflix_df['count'] = 1

# Mapping the original ratings into simplified categories
rating_mapping = {
    'TV-MA': 'Adults', 
    'R': 'Adults', 
    'NC-17': 'Adults', 
    'TV-14': 'Young adults and teens',
    'PG-13': 'Young adults and teens',
    'TV-PG': 'Families and children',
    'TV-Y7': 'Families and children',
    'TV-Y': 'Families and children',
    'G': 'Families and children',
    'TV-G': 'Families and children',
    'NR': 'Not rated',
    'TV-Y7-FV': 'Families and children',
    'UR': 'Not rated'
}

# Add a 'category' column that maps the ratings into simplified categories
netflix_df['category'] = netflix_df['rating'].map(rating_mapping)

# Group the data by category and type (Movie or TV Show)
category_counts = netflix_df.groupby(['category', 'type'])['count'].sum().unstack().fillna(0)

# Create the figure with subplots
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],  # Set width ratio (70% for the graph, 30% for the table)
    subplot_titles=['Rating Distribution by Film & TV Show', 'Rating Categories Reference'],
    horizontal_spacing=0.1,
    row_heights=[1],
    specs=[[{"type": "bar"}, {"type": "table"}]]  # Define the second column as a table
)

# Add bars for Movies
fig.add_trace(go.Bar(
    x=category_counts.index,
    y=category_counts['Movie'],
    name='Movie',
    marker=dict(color='#b20710'),
    text=category_counts['Movie'],
    textposition='outside',
    hoverinfo='x+y'
), row=1, col=1)

# Add bars for TV Shows
fig.add_trace(go.Bar(
    x=category_counts.index,
    y=category_counts['TV Show'],
    name='TV Show',
    marker=dict(color='#221f1f'),
    text=category_counts['TV Show'],
    textposition='outside',
    hoverinfo='x+y'
), row=1, col=1)

# Create the reference table with categories and their respective subcategories
rating_table = []
# We will show 4 main categories: Adults, Young adults and teens, Families and children, Not rated
main_categories = ['Adults', 'Young adults and teens', 'Families and children', 'Not rated']

for category in main_categories:
    subcategories = [key for key, value in rating_mapping.items() if value == category]
    # Add a single row for each main category with its subcategories
    rating_table.append([category, ', '.join(subcategories)])

# Add the table as a subplot with some styling
fig.add_trace(go.Table(
    header=dict(
        values=['Rating Category', 'Subgroups'],
        align='center',
        font=dict(size=14, color='white'),
        fill=dict(color='#2b2b2b'),
    ),
    cells=dict(
        values=list(zip(*rating_table)),
        align='left', 
        font=dict(size=12, color='black'),
        fill=dict(color=['#f5f5f5', '#ffffff']),
        height=30,
    ),
), row=1, col=2)

fig.update_layout(
    height=700,
    title='Rating Distribution by Film & TV Show',
    title_font=dict(size=30),
    xaxis=dict(
        title='Rating Category', 
        tickangle=45,
    ),
    yaxis=dict(
        title='Number of Collaborations',
        showticklabels=True
    ),
    legend=dict(
        title='Type',
    ),
    font=dict(size=16),
    showlegend=True,
    margin=dict(r=100)
)

fig.show()

### Movie Ratings (MPAA - Motion Picture Association of America)

- **G (General Audiences)**: Suitable for all ages. No content that parents might find inappropriate for young children.
- **PG (Parental Guidance)**: Some material may not be suitable for children. Parents are urged to provide “parental guidance” and may consider some content inappropriate for children under 10.
- **PG-13 (Parents Strongly Cautioned)**: Some material may be inappropriate for children under 13. It might contain moderate violence, language, or suggestive themes.
- **R (Restricted)**: Restricted to viewers 17 and older without an accompanying adult. Contains adult material, such as intense violence, language, drug use, or sexual content.
- **NC-17 (No One 17 and Under Admitted)**: Content specifically for adults. May contain explicit sexual content, violence, or other mature themes unsuitable for minors.

### TV Ratings (TV Parental Guidelines)

- **TV-Y (All Children)**: Designed for a very young audience, generally under age 6. Safe for all ages and free of any content that could scare young children.
- **TV-Y7 (Directed to Older Children)**: Suitable for children age 7 and older. May contain mild fantasy violence or other themes that require a bit more maturity to understand.
- **TV-Y7-FV (Fantasy Violence)**: Specifically denotes TV-Y7 content that includes fantasy violence, like superhero battles, aimed at children age 7 and older.
- **TV-G (General Audience)**: Suitable for all ages. Similar to a G-rated movie, but made for TV. Contains minimal or no mature themes.
- **TV-PG (Parental Guidance Suggested)**: Some material may not be suitable for children. May contain mild violence, language, or suggestive dialogue.
- **TV-14 (Parents Strongly Cautioned)**: Intended for viewers 14 and older. Contains material that many parents would find inappropriate for children under 14, such as moderate violence, sexual content, or strong language.
- **TV-MA (Mature Audiences)**: For mature audiences only, typically 17 or older. Contains explicit sexual content, strong language, or graphic violence.

### Other Ratings

- **NR (Not Rated)**: Content hasn’t been officially rated by an agency.
- **UR (Unrated)**: Often used for additional or director’s cuts that differ from the original rated version and have not gone through the official rating process.


## 8° Graph

### Top 20 Most Collaborative Countries in Netflix Productions

This bar chart displays the top 20 countries with the highest number of collaborative productions on Netflix. The data represents films and series produced jointly by multiple countries, highlighting the frequency of each country's involvement. By examining these collaborations, we gain insights into Netflix's international production landscape and the countries most engaged in joint film and series projects.<br>
The analysis was conducted by splitting and counting each country involved in multi-country productions, with the top 20 most frequent collaborators shown in descending order.

In [14]:
# Filter rows where 'country' contains more than one country
multi_country_productions = netflix_df[netflix_df['country'].str.contains(',', na=False)].copy()

# Split the 'country' column into lists
multi_country_productions['country'] = multi_country_productions['country'].str.split(',')

# Explode the 'country' column to create one row per country, separating multi-country productions
exploded_countries = multi_country_productions.explode('country')

# Remove any leading or trailing whitespace from country names
exploded_countries['country'] = exploded_countries['country'].str.strip()

# Count the occurrences of each country
country_collab_counts = exploded_countries['country'].value_counts()

top_20_countries = country_collab_counts.head(20)

fig = px.bar(
    top_20_countries,
    x=top_20_countries.values,
    y=top_20_countries.index,
    orientation='h',
    labels={'x': 'Number of Collaborations'},
    title='Top 20 Countries with more collaborations in Netflix Productions',
    color_discrete_sequence=['#b20710']
)

fig.update_layout(
    height=700,
    title_font=dict(size=30),
    xaxis=dict(
        title='Number of Collaborations'
    ),
    yaxis=dict(
        title='Countries',
        categoryorder='total ascending',
        tickmode='linear'
    ),
    legend=dict(
        title='Type',
    ),
    font=dict(size=16)
)

fig.show()

## 9° Graph

### 10 most frequent country pairs involved in Netflix productions

This bar chart analyze collaborations between countries, observing which pairs of nations collaborate the most in the production of content.
<br><br>
Providing insights into global partnerships and cross-border cooperation within the entertainment industry, showcasing the international nature of Netflix's production strategy.

In [16]:
from itertools import combinations

# Filter productions involving more than one country
multi_country_productions = netflix_df[netflix_df['country'].str.contains(',', na=False)]

# Remove extra spaces from country names
multi_country_productions['country'] = multi_country_productions['country'].str.split(',')
multi_country_productions['country'] = multi_country_productions['country'].apply(lambda x: [country.strip() for country in x])

country_combinations = []

# Create all country pairs
for countries in multi_country_productions['country']:
    country_pairs = list(combinations(sorted(countries), 2))
    country_combinations.extend(country_pairs)

# Calculate the frequency of each country pair
country_pair_counts = pd.Series(country_combinations).value_counts()
top_10_country_pairs = country_pair_counts.head(10)

# Create the bar chart
fig = px.bar(
    top_10_country_pairs,
    x=top_10_country_pairs.values,
    y=[f"{pair[0]} - {pair[1]}" for pair in top_10_country_pairs.index],
    orientation='h',
    labels={'x': 'Number of Collaborations', 'y': 'Country Pair'},
    title='Top 10 Country Pairs for Collaborations in Netflix Productions'
)

# Update layout and set the order of the country pairs based on their total collaboration count
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    template="plotly_white"
)

# Display the figure
fig.show()

C:\Users\lucaf\AppData\Local\Temp\ipykernel_33892\3815904265.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\lucaf\AppData\Local\Temp\ipykernel_33892\3815904265.py:8: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

